## Construct quadratic program

In [16]:
import numpy as np
import networkx as nx

from scipy.optimize import minimize

from qiskit.primitives import (
    BackendEstimator,
    BackendEstimatorV2,
    StatevectorEstimator,
    Estimator,
)
from qiskit.circuit.library import EfficientSU2
from qiskit_optimization.applications import Maxcut
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_algorithms import NumPyMinimumEigensolver, VQE
from qiskit_algorithms.optimizers import NELDER_MEAD, COBYLA
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import (
    Estimator as AerEstimator,
    EstimatorV2 as AerEstimatorV2,
)

from lcc import LCCVQE

In [17]:
n = 10
G = nx.random_regular_graph(3, n, seed=123)
max_cut = Maxcut(G)
qp_max_cut = max_cut.to_quadratic_program()
print(qp_max_cut.prettyprint())

Problem name: Max-cut

Maximize
  -2*x_0*x_1 - 2*x_0*x_3 - 2*x_0*x_8 - 2*x_1*x_8 - 2*x_1*x_9 - 2*x_2*x_3
  - 2*x_2*x_7 - 2*x_2*x_8 - 2*x_3*x_6 - 2*x_4*x_5 - 2*x_4*x_6 - 2*x_4*x_7
  - 2*x_5*x_7 - 2*x_5*x_9 - 2*x_6*x_9 + 3*x_0 + 3*x_1 + 3*x_2 + 3*x_3 + 3*x_4
  + 3*x_5 + 3*x_6 + 3*x_7 + 3*x_8 + 3*x_9

Subject to
  No constraints

  Binary variables (10)
    x_0 x_1 x_2 x_3 x_4 x_5 x_6 x_7 x_8 x_9



In [18]:
exact = MinimumEigenOptimizer(NumPyMinimumEigensolver())
result_max_cut = exact.solve(qp_max_cut)
print(result_max_cut.prettyprint())

objective function value: 13.0
variable values: x_0=0.0, x_1=0.0, x_2=0.0, x_3=1.0, x_4=1.0, x_5=0.0, x_6=0.0, x_7=1.0, x_8=1.0, x_9=1.0
status: SUCCESS


## Solve using LCC

In [40]:
estimator = StatevectorEstimator()
optimizer = COBYLA()

solver = LCCVQE(
    qp_max_cut,
    estimator=estimator,
    optimizer=optimizer,
)

initial_params = np.random.rand(2 * qp_max_cut.get_num_vars()) * 2 * np.pi
print('Initial params:', initial_params.view())
solver.compute_energy(initial_params)

Initial params: [1.1056335  5.46670057 5.25751333 0.29044648 5.53028028 0.76883826
 2.95930335 3.48675864 4.88791505 0.2136711  4.82761384 3.61246601
 5.42791484 1.0456249  0.63676249 3.69102887 5.88313965 5.96231445
 5.94680802 6.05199716]


-0.7244063053914535

In [41]:
ansatz = EfficientSU2(
    num_qubits=qp_max_cut.get_num_vars(),
    su2_gates='ry',
    entanglement='circular',
    reps=1
)
result = estimator.run([(ansatz, solver.hamiltonian, initial_params)]).result()[0]
result.data.evs

array(-0.95633795)

In [44]:
ansatz.decompose().draw(fold=-1)

┌──────────┐┌───┐     ┌───────────┐                                                                                                        
q_0: ┤ Ry(θ[0]) ├┤ X ├──■──┤ Ry(θ[10]) ├────────────────────────────────────────────────────────────────────────────────────────────────────────
     ├──────────┤└─┬─┘┌─┴─┐└───────────┘┌───────────┐                                                                                           
q_1: ┤ Ry(θ[1]) ├──┼──┤ X ├──────■──────┤ Ry(θ[11]) ├───────────────────────────────────────────────────────────────────────────────────────────
     ├──────────┤  │  └───┘    ┌─┴─┐    └───────────┘┌───────────┐                                                                              
q_2: ┤ Ry(θ[2]) ├──┼───────────┤ X ├──────────■──────┤ Ry(θ[12]) ├──────────────────────────────────────────────────────────────────────────────
     ├──────────┤  │           └───┘        ┌─┴─┐    └───────────┘┌───────────┐                                                                 
q_3: ┤ Ry(θ[3]) ├──┼────────────────────────┤ X ├──────────■──────┤ Ry(θ[13]) ├─────────────────────────────────────────────────────────────────
     ├──────────┤  │                        └───┘        ┌─┴─┐    └───────────┘┌───────────┐                                                    
q_4: ┤ Ry(θ[4]) ├──┼─────────────────────────────────────┤ X ├──────────■──────┤ Ry(θ[14]) ├────────────────────────────────────────────────────
     ├──────────┤  │                                     └───┘        ┌─┴─┐    └───────────┘┌───────────┐                                       
q_5: ┤ Ry(θ[5]) ├──┼──────────────────────────────────────────────────┤ X ├──────────■──────┤ Ry(θ[15]) ├───────────────────────────────────────
     ├──────────┤  │                                                  └───┘        ┌─┴─┐    └───────────┘┌───────────┐                          
q_6: ┤ Ry(θ[6]) ├──┼───────────────────────────────────────────────────────────────┤ X ├──────────■──────┤ Ry(θ[16]) ├──────────────────────────
     ├──────────┤  │                                                               └───┘        ┌─┴─┐    └───────────┘┌───────────┐             
q_7: ┤ Ry(θ[7]) ├──┼────────────────────────────────────────────────────────────────────────────┤ X ├──────────■──────┤ Ry(θ[17]) ├─────────────
     ├──────────┤  │                                                                            └───┘        ┌─┴─┐    └───────────┘┌───────────┐
q_8: ┤ Ry(θ[8]) ├──┼─────────────────────────────────────────────────────────────────────────────────────────┤ X ├──────────■──────┤ Ry(θ[18]) ├
     ├──────────┤  │                                                                                         └───┘        ┌─┴─┐    ├───────────┤
q_9: ┤ Ry(θ[9]) ├──■──────────────────────────────────────────────────────────────────────────────────────────────────────┤ X ├────┤ Ry(θ[19]) ├
     └──────────┘                                                                                                         └───┘    └───────────┘

In [ ]:
# estimator = Estimator()
# optimizer = COBYLA()

# solver = LCCVQE(
#     qp_max_cut,
#     estimator=estimator,
#     optimizer=optimizer,
# )

# initial_params = np.random.rand(2 * qp_max_cut.get_num_vars()) * 2 * np.pi
# print('Initial params:', initial_params.view())
# solver.compute_energy_v1(initial_params)

In [30]:
result = solver.solve(initial_params)
print(result)

{   'fun': -5.499999976558985,
    'jac': None,
    'nfev': 609,
    'nit': None,
    'njev': None,
    'x': array([ 1.57072920e+00,  4.71241237e+00,  4.71233687e+00,  6.28314886e+00,
        3.14154203e+00,  3.14163338e+00,  3.14156281e+00,  6.28321839e+00,
        3.14163920e+00, -4.78022774e-05,  4.71242920e+00,  4.71235302e+00,
        3.14164618e+00,  2.09998859e-05,  3.14169516e+00,  3.14160989e+00,
        3.48289916e-06,  3.14167081e+00,  3.14161216e+00,  2.93179820e-05])}


In [31]:
print('Expectation:', -(result.fun + solver.offset))

Expectation: 12.999999976558986


In [44]:
solver.offset

-7.5

## Solve without LCC

In [32]:
ham, offset = qp_max_cut.to_ising()

backend = AerSimulator(method='statevector')
estimator = Estimator()
ansatz = EfficientSU2(
    num_qubits=qp_max_cut.get_num_vars(),
    su2_gates='ry',
    entanglement='circular',
    reps=1
)
vqe = VQE(estimator, ansatz, optimizer, initial_point=initial_params)
result = vqe.compute_minimum_eigenvalue(ham)
print(result)

{   'aux_operators_evaluated': None,
    'cost_function_evals': 690,
    'eigenvalue': -5.499999990644686,
    'optimal_circuit': <qiskit.circuit.library.n_local.efficient_su2.EfficientSU2 object at 0x7f671f9fa420>,
    'optimal_parameters': {   ParameterVectorElement(θ[0]): 2.472524493447961,
                              ParameterVectorElement(θ[12]): 3.1415897978213305,
                              ParameterVectorElement(θ[18]): 6.2831915042113655,
                              ParameterVectorElement(θ[13]): 9.195194891428881e-06,
                              ParameterVectorElement(θ[11]): 3.141657144709545,
                              ParameterVectorElement(θ[10]): 6.283169152199935,
                              ParameterVectorElement(θ[19]): -2.712147511078236e-05,
                              ParameterVectorElement(θ[9]): -2.1962060569801105e-06,
                              ParameterVectorElement(θ[16]): 4.4147768729730766e-05,
                              ParameterVecto

In [111]:
result = estimator.run([(ansatz, ham, initial_params)]).result()[0]
result.data.evs

array(-0.36011752)

In [15]:
result = estimator.run(ansatz, ham, initial_params).result()
result.values[0]

-0.319783665312153